# 📊 UI Index Forensic Analysis

**The Data Vigilante | Comparative Static Model**

This notebook analyzes the documented decay of unemployment insurance safety net adequacy across DC, Maryland, and Virginia from 2010 to 2026.

## Metrics
- **BAI** (Benefit Adequacy Index): Max WBA / Weekly Housing Cost
- **WBI** (Regressive Wage Base Index): Taxable Wage Base / Avg Annual Wage
- **MIPI** (Multi-Income Penalty Index): Clawback severity at $250 side-hustle earnings
- **Housing Gap**: Weekly Housing - Max WBA (survival deficit)

In [ ]:
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 150

DATA_PATH = Path('data/dmv_macro_baselines.csv')
records = []
with open(DATA_PATH, 'r', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append({
            'Jurisdiction': row['Jurisdiction'],
            'Year': int(row['Year']),
            'Max_WBA': float(row['Max_WBA']),
            'Taxable_Wage_Base': float(row['Taxable_Wage_Base']),
            'Avg_Annual_Wage': float(row['Avg_Annual_Wage']),
            'Weekly_Housing': float(row['Weekly_Housing']),
        })

for rec in records:
    rec['BAI'] = rec['Max_WBA'] / rec['Weekly_Housing']
    rec['WBI'] = rec['Taxable_Wage_Base'] / rec['Avg_Annual_Wage']
    net_counted = max(0, 250 - 50)
    rec['MIPI'] = net_counted / rec['Max_WBA']
    rec['Housing_Gap'] = rec['Weekly_Housing'] - rec['Max_WBA']

df = pd.DataFrame(records)
df.head(12)

## Chart 1: BAI Decay Trajectory

Shows how benefit adequacy eroded across all three jurisdictions. The red threshold line at 1.0 marks the survival boundary — any index below this means the maximum weekly benefit cannot cover median housing costs.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for jurisdiction in df['Jurisdiction'].unique():
    sub = df[df['Jurisdiction'] == jurisdiction].sort_values('Year')
    ax.plot(sub['Year'], sub['BAI'], marker='o', linewidth=2.5, markersize=8, label=jurisdiction)

ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Adequacy Threshold (1.0)')
ax.set_xlabel('Year')
ax.set_ylabel('Benefit Adequacy Index (BAI)')
ax.set_title('BAI Decay Trajectory: 2010 → 2026')
ax.legend(title='Jurisdiction')
ax.set_ylim(0.5, 1.5)
plt.tight_layout()
plt.show()

## Chart 2: WBI Stagnation

The Regressive Wage Base Index reveals how flat SUI taxable wage caps (MD frozen at $8,500 since 1992) become progressively smaller fractions of average wages over time.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for jurisdiction in df['Jurisdiction'].unique():
    sub = df[df['Jurisdiction'] == jurisdiction].sort_values('Year')
    ax.plot(sub['Year'], sub['WBI'], marker='s', linewidth=2.5, markersize=8, label=jurisdiction)

ax.set_xlabel('Year')
ax.set_ylabel('Regressive Wage Base Index (WBI)')
ax.set_title('WBI Stagnation: Flat SUI Caps vs. Rising Wages')
ax.legend(title='Jurisdiction')
plt.tight_layout()
plt.show()

## Chart 3: MIPI Clawback Severity

The Multi-Income Penalty Index measures the institutional penalty applied to workers who supplement UI benefits with part-time work. Higher MIPI = greater clawback.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for jurisdiction in df['Jurisdiction'].unique():
    sub = df[df['Jurisdiction'] == jurisdiction].sort_values('Year')
    ax.plot(sub['Year'], sub['MIPI'], marker='^', linewidth=2.5, markersize=8, label=jurisdiction)

ax.set_xlabel('Year')
ax.set_ylabel('Multi-Income Penalty Index (MIPI)')
ax.set_title('MIPI Clawback Severity at $250 Side-Hustle Earnings')
ax.legend(title='Jurisdiction')
plt.tight_layout()
plt.show()

## Chart 4: Housing Cost vs. WBA Gap

Direct comparison of maximum weekly benefits against median weekly housing costs. The gap represents the survival deficit a UI recipient must cover from other sources.

In [ ]:
import numpy as np

fig, ax = plt.subplots(figsize=(12, 6))
years = sorted(df['Year'].unique())
jurisdictions = df['Jurisdiction'].unique()
x = np.arange(len(years))
width = 0.25

for i, jurisdiction in enumerate(jurisdictions):
    sub = df[df['Jurisdiction'] == jurisdiction].sort_values('Year')
    wba_vals = sub['Max_WBA'].values
    housing_vals = sub['Weekly_Housing'].values
    offset = (i - 1) * width
    ax.bar(x + offset - width/2, wba_vals, width, label=f'{jurisdiction} WBA', alpha=0.9)
    ax.bar(x + offset + width/2, housing_vals, width, label=f'{jurisdiction} Housing', alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels(years)
ax.set_xlabel('Year')
ax.set_ylabel('Weekly Amount ($)')
ax.set_title('Housing Cost vs. Maximum Weekly Benefit: The Growing Gap')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Key Findings Summary

| Jurisdiction | BAI 2010 | BAI 2026 | Δ BAI | Direction |
|-------------|----------|----------|-------|-----------|
| Maryland | 1.46 | 0.96 | -0.50 | **WORSENING** |
| Virginia | 1.40 | 0.90 | -0.50 | **WORSENING** |
| DC | 0.94 | 0.85 | -0.09 | **WORSENING** |

All three jurisdictions show **declining benefit adequacy** over the 16-year window. Maryland and Virginia crossed below the survival threshold (1.0) by 2026. DC was already below threshold in 2010 and continues to deteriorate.